In [6]:
# -*- coding: utf-8 -*-
import arcpy
import os

arcpy.env.overwriteOutput = True

# =========================
# 1) 使用者需要修改的參數
# =========================
fc_A = r"D:\ArcGIS\Morpho_Spatial_Demography\MSD_Taiwan.gdb\OL_P1_2204_corals"          # 舊調查 (A)
fc_B = r"D:\ArcGIS\Morpho_Spatial_Demography\MSD_Taiwan.gdb\OL_P1_2209_cor_SimplifyPolyg"  # 新調查 (B) -> target

A_id_field   = "Temp_ID"   # A 的 ID 欄位
A_form_field = "Forms"     # A 的 Forms 欄位
B_form_field = "Forms"     # B 的 Forms 欄位

B_prev_id_field = "Prev_ID"  # 寫入到 B 的欄位（存 A 的 ID）

out_gdb = os.path.dirname(fc_B)
sj_fc = os.path.join(out_gdb, "B_joinA_largestOverlap")

# =========================
# 2) 確保 B 有輸出欄位
# =========================
b_fields = [f.name for f in arcpy.ListFields(fc_B)]
if B_prev_id_field not in b_fields:
    arcpy.AddField_management(fc_B, B_prev_id_field, "TEXT", field_length=80)

# =========================
# 3) Spatial Join：B (target) + A (join) , Largest Overlap
# =========================
arcpy.analysis.SpatialJoin(
    target_features=fc_B,
    join_features=fc_A,
    out_feature_class=sj_fc,
    join_operation="JOIN_ONE_TO_ONE",
    join_type="KEEP_ALL",
    match_option="LARGEST_OVERLAP"
)

sj_names = [f.name for f in arcpy.ListFields(sj_fc)]

def pick_field_for_target(base_name):
    """target=B 欄位：優先 base_name，其次 base_name_1"""
    if base_name in sj_names:
        return base_name
    if f"{base_name}_1" in sj_names:
        return f"{base_name}_1"
    raise RuntimeError(f"[Target field missing] 找不到 {base_name} 或 {base_name}_1")

def pick_field_for_join(base_name):
    """join=A 欄位：優先 base_name_1，其次 base_name"""
    if f"{base_name}_1" in sj_names:
        return f"{base_name}_1"
    if base_name in sj_names:
        return base_name
    raise RuntimeError(f"[Join field missing] 找不到 {base_name}_1 或 {base_name}")

# 正確抓欄位：A 用 _1，B 用原名
sj_A_id_field   = pick_field_for_join(A_id_field)
sj_A_form_field = pick_field_for_join(A_form_field)
sj_B_form_field = pick_field_for_target(B_form_field)

# TARGET_FID 一定要有，用來回寫到 B
if "TARGET_FID" not in sj_names:
    raise RuntimeError("找不到 TARGET_FID，無法回寫到 B。")
target_fid_field = "TARGET_FID"

def norm(x):
    if x is None:
        return None
    return str(x).strip().lower()

# =========================
# 4) 建立 write_map：用 A_id 是否為空來判斷有沒有 join（不再看 Join_Count）
# =========================
write_map = {}
total_B = int(arcpy.management.GetCount(fc_B)[0])

sj_rows = 0
success = 0
lost_no_join = 0
lost_form_mismatch = 0

with arcpy.da.SearchCursor(
    sj_fc,
    [target_fid_field, sj_A_id_field, sj_A_form_field, sj_B_form_field]
) as cur:
    for b_oid, a_id, a_form, b_form in cur:
        sj_rows += 1

        # 沒 join（或 join 到的 A 沒有 ID）
        if a_id in (None, ""):
            write_map[b_oid] = None
            lost_no_join += 1
            continue

        # Form 必須相同才寫入
        if norm(a_form) == norm(b_form):
            write_map[b_oid] = str(a_id)
            success += 1
        else:
            write_map[b_oid] = None
            lost_form_mismatch += 1

# =========================
# 5) 回寫到原始 B
# =========================
updated = 0
with arcpy.da.UpdateCursor(fc_B, ["OID@", B_prev_id_field]) as ucur:
    for oid, prev in ucur:
        if oid not in write_map:
            continue
        new_val = write_map[oid]

        # 想保留舊值就把下面清空那段註解掉
        if new_val is None:
            if prev not in (None, ""):
                ucur.updateRow([oid, None])
                updated += 1
        else:
            if prev != new_val:
                ucur.updateRow([oid, new_val])
                updated += 1

# =========================
# 6) 回報
# =========================
lost_total = lost_no_join + lost_form_mismatch

# =========================
# 7) 清除 Spatial Join 暫存圖層（穩定版）
# =========================
try:
    aprx = arcpy.mp.ArcGISProject("CURRENT")
    sj_path = sj_fc.lower()
    removed_count = 0

    for m in aprx.listMaps():
        for lyr in m.listLayers():
            # 先移除 broken layer（通常就是你剛刪掉資料後留下的紅驚嘆號）
            try:
                if getattr(lyr, "isBroken", False):
                    m.removeLayer(lyr)
                    removed_count += 1
                    continue
            except:
                pass

            # 再移除 dataSource 指向 sj_fc 的 layer
            try:
                if hasattr(lyr, "dataSource") and lyr.dataSource.lower() == sj_path:
                    m.removeLayer(lyr)
                    removed_count += 1
            except:
                # 有些 layer 取 dataSource 也會噴錯，略過即可
                pass

    print(f"[CLEAN] Removed layers from Contents: {removed_count}")

except Exception as e:
    print(f"[WARN] Could not clean layers from map: {e}")

# 刪除 GDB 中的 feature class
try:
    if arcpy.Exists(sj_fc):
        arcpy.management.Delete(sj_fc)
        print(f"[CLEAN] Deleted feature class: {sj_fc}")
except Exception as e:
    print(f"[ERROR] Failed to delete spatial join output: {e}")


print("====================================")
print("[RESULT] Largest Overlap + Form match -> write A Temp_ID into B Prev_ID")
print(f"Total B features                : {total_B}")
print(f"SpatialJoin output rows         : {sj_rows}")
print(f"Success (A_ID written)          : {success}")
print(f"Lost - no join (A_ID is null)   : {lost_no_join}")
print(f"Lost - form mismatch            : {lost_form_mismatch}")
print(f"Lost total                      : {lost_total}")
print(f"Rows actually updated in B      : {updated}")
print(f"SpatialJoin FC saved as         : {sj_fc}")
print(f"Used fields -> A_id: {sj_A_id_field}, A_form: {sj_A_form_field}, B_form: {sj_B_form_field}")
print("====================================")


[CLEAN] Removed layers from Contents: 4
[CLEAN] Deleted feature class: D:\ArcGIS\Morpho_Spatial_Demography\MSD_Taiwan.gdb\B_joinA_largestOverlap
[RESULT] Largest Overlap + Form match -> write A Temp_ID into B Prev_ID
Total B features                : 1106
SpatialJoin output rows         : 1106
Success (A_ID written)          : 826
Lost - no join (A_ID is null)   : 126
Lost - form mismatch            : 154
Lost total                      : 280
Rows actually updated in B      : 0
SpatialJoin FC saved as         : D:\ArcGIS\Morpho_Spatial_Demography\MSD_Taiwan.gdb\B_joinA_largestOverlap
Used fields -> A_id: Temp_ID, A_form: Forms_1, B_form: Forms
